# 1.1 DataFrame Creation and Structure ? Oracle HR Practice

**Hands-on PySpark notebook | Oracle HR sample-schema domain**

All records are synthetic and follow Oracle HR table and column conventions. The seven
reusable DataFrames have valid primary/foreign-key references where keys are present.
Intentional data-quality cases include a repeated employee email, NULL manager/department/
commission values, and incomplete location attributes; primary keys remain unique.

**Learning objectives**
- Create PySpark DataFrames from HR records and inspect their structures.
- Validate column names, data types, and nullability for HR migration contracts.
- Profile compensation and job ranges with built-in DataFrame summary APIs.
- Preserve HR keys, NULLs, and duplicate business identifiers during preparation.

## 1. PySpark environment setup

    Target: PySpark 3.5+ with a compatible Java runtime. Explicit Spark schemas and built-in
    DataFrame APIs are used; Pandas is not used.

## 2. Required imports

In [1]:
from datetime import date
from decimal import Decimal
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import DateType, DecimalType, IntegerType, StringType, StructField, StructType

## 3. SparkSession creation

In [2]:
spark = (SparkSession.builder.appName("Oracle HR DataFrame Creation and Structure Practice")
         .master("local[2]").getOrCreate())
spark.sparkContext.setLogLevel("WARN")
print("Spark version:", spark.version)

Spark version: 3.5.5


## 4. Create reusable Oracle HR DataFrames

    EMPLOYEES and DEPARTMENTS contain 20 and 18 records. Related job, location, country, region,
    and job-history records are included. A repeated employee email is intentional; primary keys
    remain unique.

In [3]:
region_schema = StructType([StructField("region_id", IntegerType(), False), StructField("region_name", StringType(), False)])
regions_df = spark.createDataFrame([
    (1, "Europe"), (2, "Americas"), (3, "Asia"), (4, "Middle East and Africa"), (5, "Oceania")
], region_schema)

country_schema = StructType([
    StructField("country_id", StringType(), False), StructField("country_name", StringType(), False),
    StructField("region_id", IntegerType(), False)
])
countries_df = spark.createDataFrame([
    ("US", "United States of America", 2), ("CA", "Canada", 2), ("BR", "Brazil", 2),
    ("GB", "United Kingdom", 1), ("DE", "Germany", 1), ("FR", "France", 1),
    ("IN", "India", 3), ("JP", "Japan", 3), ("AU", "Australia", 5), ("EG", "Egypt", 4)
], country_schema)

location_schema = StructType([
    StructField("location_id", IntegerType(), False), StructField("street_address", StringType(), True),
    StructField("postal_code", StringType(), True), StructField("city", StringType(), False),
    StructField("state_province", StringType(), True), StructField("country_id", StringType(), False)
])
locations_df = spark.createDataFrame([
    (1000, "200 Market Street", "94105", "San Francisco", "California", "US"),
    (1001, "500 Fifth Avenue", "10018", "New York", "New York", "US"),
    (1002, "77 Massachusetts Avenue", "02139", "Cambridge", "Massachusetts", "US"),
    (1003, "10 King Street", "M5H 1A1", "Toronto", "Ontario", "CA"),
    (1004, "120 Rua Augusta", "01305-000", "Sao Paulo", "Sao Paulo", "BR"),
    (1005, "25 Canada Square", "E14 5LQ", "London", None, "GB"),
    (1006, "Unter den Linden 5", "10117", "Berlin", None, "DE"),
    (1007, "8 Rue de Rivoli", "75001", "Paris", None, "FR"),
    (1008, "14 Residency Road", "560025", "Bengaluru", "Karnataka", "IN"),
    (1009, "6 Connaught Place", "110001", "New Delhi", "Delhi", "IN"),
    (1010, "33 Marine Drive", "400020", "Mumbai", "Maharashtra", "IN"),
    (1011, "1 Chome Marunouchi", "100-0005", "Tokyo", None, "JP"),
    (1012, "88 George Street", "2000", "Sydney", "New South Wales", "AU"),
    (1013, "22 Nile Corniche", None, "Cairo", None, "EG"),
    (1014, "400 Lake Shore Drive", "60611", "Chicago", "Illinois", "US"),
    (1015, "9 Market Street", "94105", "San Francisco", "California", "US"),
    (1016, "70 Fleet Street", "EC4Y 1EU", "London", None, "GB"),
    (1017, "15 Park Street", None, "Kolkata", "West Bengal", "IN")
], location_schema)

job_schema = StructType([
    StructField("job_id", StringType(), False), StructField("job_title", StringType(), False),
    StructField("min_salary", DecimalType(8, 2), False), StructField("max_salary", DecimalType(8, 2), False)
])
jobs_df = spark.createDataFrame([
    ("AD_PRES", "President", Decimal("20000.00"), Decimal("40000.00")),
    ("AD_VP", "Administration Vice President", Decimal("15000.00"), Decimal("30000.00")),
    ("IT_PROG", "Programmer", Decimal("4000.00"), Decimal("12000.00")),
    ("FI_MGR", "Finance Manager", Decimal("8000.00"), Decimal("16000.00")),
    ("FI_ACCOUNT", "Accountant", Decimal("4000.00"), Decimal("10000.00")),
    ("PU_MAN", "Purchasing Manager", Decimal("8000.00"), Decimal("15000.00")),
    ("PU_CLERK", "Purchasing Clerk", Decimal("2000.00"), Decimal("5000.00")),
    ("SA_MAN", "Sales Manager", Decimal("10000.00"), Decimal("20000.00")),
    ("SA_REP", "Sales Representative", Decimal("5000.00"), Decimal("12000.00")),
    ("HR_REP", "Human Resources Representative", Decimal("4000.00"), Decimal("9000.00")),
    ("HR_MGR", "Human Resources Manager", Decimal("8000.00"), Decimal("15000.00")),
    ("MK_MAN", "Marketing Manager", Decimal("9000.00"), Decimal("15000.00")),
    ("MK_REP", "Marketing Representative", Decimal("5000.00"), Decimal("11000.00")),
    ("ST_MAN", "Stock Manager", Decimal("5000.00"), Decimal("10000.00")),
    ("ST_CLERK", "Stock Clerk", Decimal("2000.00"), Decimal("5000.00")),
    ("AC_MGR", "Accounting Manager", Decimal("8000.00"), Decimal("16000.00")),
    ("AC_ACCOUNT", "Public Accountant", Decimal("4000.00"), Decimal("10000.00")),
    ("PR_REP", "Public Relations Representative", Decimal("5000.00"), Decimal("11000.00"))
], job_schema)

employee_schema = StructType([
    StructField("employee_id", IntegerType(), False), StructField("first_name", StringType(), True),
    StructField("last_name", StringType(), False), StructField("email", StringType(), True),
    StructField("phone_number", StringType(), True), StructField("hire_date", DateType(), False),
    StructField("job_id", StringType(), False), StructField("salary", DecimalType(8, 2), False),
    StructField("commission_pct", DecimalType(4, 2), True), StructField("manager_id", IntegerType(), True),
    StructField("department_id", IntegerType(), True)
])
employees_df = spark.createDataFrame([
    (100, "Olivia", "Bennett", "OBENNETT", "515.123.1000", date(1987,6,17), "AD_PRES", Decimal("24000.00"), None, None, 90),
    (101, "Marcus", "Chen", "MCHEN", "515.123.1001", date(1989,9,21), "AD_VP", Decimal("17000.00"), None, 100, 90),
    (102, "Priya", "Nair", "PNAIR", "515.123.1002", date(1993,1,13), "AD_VP", Decimal("17000.00"), None, 100, 90),
    (103, "Daniel", "Brooks", "DBROOKS", "590.423.1003", date(1990,1,3), "IT_PROG", Decimal("9000.00"), None, 102, 60),
    (104, "Sofia", "Patel", "SPATEL", "590.423.1004", date(1991,5,21), "IT_PROG", Decimal("6000.00"), None, 103, 60),
    (105, "Ethan", "Morris", "EMORRIS", "590.423.1005", date(1997,6,25), "IT_PROG", Decimal("4800.00"), None, 103, 60),
    (106, "Aisha", "Khan", "AKHAN", "590.423.1006", date(1998,2,5), "IT_PROG", Decimal("4800.00"), None, 103, 60),
    (107, "Lucas", "Meyer", "LMEYER", "590.423.1007", date(1999,2,7), "IT_PROG", Decimal("4200.00"), None, 103, 60),
    (108, "Grace", "Kim", "GKIM", "515.124.1008", date(1994,8,17), "FI_MGR", Decimal("12000.00"), None, 101, 100),
    (109, "Noah", "Williams", "NWILLIAMS", "515.124.1009", date(1994,8,16), "FI_ACCOUNT", Decimal("9000.00"), None, 108, 100),
    (110, "Elena", "Garcia", "EGARCIA", "515.124.1010", date(1997,9,28), "FI_ACCOUNT", Decimal("8200.00"), None, 108, 100),
    (111, "Ibrahim", "Hassan", "IHASSAN", "515.124.1011", date(1997,9,30), "FI_ACCOUNT", Decimal("7700.00"), None, 108, 100),
    (112, "Chloe", "Dubois", "CDUBOIS", "515.124.1012", date(1998,3,7), "FI_ACCOUNT", Decimal("7800.00"), None, 108, 100),
    (113, "Mateo", "Silva", "MSILVA", "515.124.1013", date(1999,12,7), "FI_ACCOUNT", Decimal("6900.00"), None, 108, 100),
    (114, "Amara", "Okafor", "AOKAFOR", "515.127.1014", date(1994,12,7), "PU_MAN", Decimal("11000.00"), None, 100, 30),
    (115, "Theo", "Martin", "TMARTIN", "515.127.1015", date(1995,5,18), "PU_CLERK", Decimal("3100.00"), None, 114, 30),
    (116, "Mei", "Tanaka", "MTANAKA", "515.127.1016", date(1997,12,24), "SA_MAN", Decimal("11000.00"), Decimal("0.15"), 100, 80),
    (117, "Fatima", "Rahman", "FRAHMAN", "515.127.1017", date(1997,7,24), "SA_REP", Decimal("8000.00"), Decimal("0.10"), 116, 80),
    (118, "Fatima", "Rahman", "FRAHMAN", "515.127.1018", date(1997,7,24), "SA_REP", Decimal("8000.00"), Decimal("0.10"), 116, 80),
    (119, "Ben", "Carter", None, None, date(1999,8,10), "HR_REP", Decimal("2500.00"), None, 100, None)
], employee_schema)

department_schema = StructType([
    StructField("department_id", IntegerType(), False), StructField("department_name", StringType(), False),
    StructField("manager_id", IntegerType(), True), StructField("location_id", IntegerType(), True)
])
departments_df = spark.createDataFrame([
    (10,"Administration",101,1000), (20,"Marketing",103,1001), (30,"Purchasing",114,1003),
    (40,"Human Resources",119,1008), (50,"Shipping",114,1009), (60,"IT",103,1002),
    (70,"Public Relations",101,1005), (80,"Sales",116,1014), (90,"Executive",100,1000),
    (100,"Finance",108,1006), (110,"Accounting Operations",109,1007), (120,"Recruiting",119,1017),
    (130,"Compensation",101,1008), (140,"Business Intelligence",103,1015),
    (150,"Customer Support",114,1016), (160,"Risk Management",108,1001),
    (170,"Legal",101,1005), (180,"Facilities",None,None)
], department_schema)

job_history_schema = StructType([
    StructField("employee_id", IntegerType(), False), StructField("start_date", DateType(), False),
    StructField("end_date", DateType(), False), StructField("job_id", StringType(), False),
    StructField("department_id", IntegerType(), True)
])
job_history_df = spark.createDataFrame([
    (101,date(1987,9,21),date(1989,9,20),"AC_ACCOUNT",110),
    (102,date(1989,1,13),date(1992,12,31),"IT_PROG",60),
    (103,date(1987,1,3),date(1989,12,31),"SA_REP",80),
    (104,date(1989,5,21),date(1991,5,20),"IT_PROG",60),
    (105,date(1994,6,25),date(1997,6,24),"IT_PROG",60),
    (108,date(1990,8,17),date(1994,8,16),"FI_ACCOUNT",100),
    (109,date(1990,8,16),date(1994,8,15),"AC_ACCOUNT",110),
    (110,date(1994,9,28),date(1997,9,27),"AC_ACCOUNT",110),
    (114,date(1990,12,7),date(1994,12,6),"PU_CLERK",30),
    (115,date(1992,5,18),date(1995,5,17),"PU_CLERK",30),
    (116,date(1993,12,24),date(1997,12,23),"SA_REP",80),
    (117,date(1994,7,24),date(1997,7,23),"SA_REP",80),
    (118,date(1994,7,24),date(1997,7,23),"SA_REP",80),
    (119,date(1995,8,10),date(1999,8,9),"HR_REP",40),
    (101,date(1985,1,1),date(1987,9,20),"AC_ACCOUNT",110)
], job_history_schema)

hr_dataframes = {
    "employees_df": employees_df, "departments_df": departments_df,
    "jobs_df": jobs_df, "job_history_df": job_history_df,
    "locations_df": locations_df, "countries_df": countries_df, "regions_df": regions_df
}
expected_counts = {"employees_df":20, "departments_df":18, "jobs_df":18,
                   "job_history_df":15, "locations_df":18, "countries_df":10, "regions_df":5}
for dataframe_name, dataframe in hr_dataframes.items():
    count = dataframe.count()
    assert count == expected_counts[dataframe_name], f"{dataframe_name}: unexpected record count {count}"
    print(f"{dataframe_name}: {count} records")

employee_ids = {row.employee_id for row in employees_df.select("employee_id").collect()}
department_ids = {row.department_id for row in departments_df.select("department_id").collect()}
job_ids = {row.job_id for row in jobs_df.select("job_id").collect()}
location_ids = {row.location_id for row in locations_df.select("location_id").collect()}
country_ids = {row.country_id for row in countries_df.select("country_id").collect()}
region_ids = {row.region_id for row in regions_df.select("region_id").collect()}
assert employees_df.select("employee_id").distinct().count() == 20
assert departments_df.select("department_id").distinct().count() == 18
assert all(r.manager_id is None or r.manager_id in employee_ids for r in employees_df.select("manager_id").collect())
assert all(r.manager_id is None or r.manager_id in employee_ids for r in departments_df.select("manager_id").collect())
assert all(r.department_id is None or r.department_id in department_ids for r in employees_df.select("department_id").collect())
assert all(r.job_id in job_ids for r in employees_df.select("job_id").collect())
assert all(r.employee_id in employee_ids and r.job_id in job_ids and (r.department_id is None or r.department_id in department_ids) for r in job_history_df.collect())
assert all(r.country_id in country_ids for r in locations_df.select("country_id").collect())
assert all(r.region_id in region_ids for r in countries_df.select("region_id").collect())
assert all(r.location_id is None or r.location_id in location_ids for r in departments_df.select("location_id").collect())
print("Primary/foreign key checks passed; duplicate employee email is intentional.")

employees_df: 20 records
departments_df: 18 records
jobs_df: 18 records
job_history_df: 15 records
locations_df: 18 records
countries_df: 10 records
regions_df: 5 records
Primary/foreign key checks passed; duplicate employee email is intentional.


## 5. Preview sample records and schemas

    Review the sample rows and declared schemas before attempting the exercises. All seven
    DataFrames remain reusable throughout the notebook.

In [4]:
for dataframe_name, dataframe in hr_dataframes.items():
    print(f"\n--- {dataframe_name}: sample rows ---")
    dataframe.show(3, truncate=False)
    print(f"--- {dataframe_name}: schema ---")
    dataframe.printSchema()


--- employees_df: sample rows ---
+-----------+----------+---------+--------+------------+----------+-------+--------+--------------+----------+-------------+
|employee_id|first_name|last_name|email   |phone_number|hire_date |job_id |salary  |commission_pct|manager_id|department_id|
+-----------+----------+---------+--------+------------+----------+-------+--------+--------------+----------+-------------+
|100        |Olivia    |Bennett  |OBENNETT|515.123.1000|1987-06-17|AD_PRES|24000.00|NULL          |NULL      |90           |
|101        |Marcus    |Chen     |MCHEN   |515.123.1001|1989-09-21|AD_VP  |17000.00|NULL          |100       |90           |
|102        |Priya     |Nair     |PNAIR   |515.123.1002|1993-01-13|AD_VP  |17000.00|NULL          |100       |90           |
+-----------+----------+---------+--------+------------+----------+-------+--------+--------------+----------+-------------+
only showing top 3 rows

--- employees_df: schema ---
root
 |-- employee_id: integer (null

## 6. Practical questions

    Each prompt is followed by one completely empty code cell for your independent solution.
    There are three practical tasks per listed concept.

## createDataFrame()

### Question 1 ? Easy

**Business scenario:** HR migration support needs a small employee extract reconstructed from the reusable employee master.

**Problem statement:** Use `employees_df` records for employee IDs 100 and 101 to create a new DataFrame with an explicit schema containing employee ID, first name, last name, job ID, and salary.

**Expected output:** A two-row DataFrame with columns `employee_id`, `first_name`, `last_name`, `job_id`, and `salary`; employee IDs are 100 and 101 and salary remains decimal.

### Question 2 ? Medium

**Business scenario:** A recovery workflow must rebuild a few employee records from the current HR master without losing nullable department assignments.

**Problem statement:** Create a DataFrame from the existing rows for employees 100, 103, and 119 using an explicit schema matching `employees_df`.

**Expected output:** Three records and the same 11 columns/types as the employee master; employee 119 retains a NULL `department_id`.

### Question 3 ? Hard

**Business scenario:** A master-data steward is isolating two employee records that share an email address for a migration-quality review.

**Problem statement:** Use `createDataFrame()` and an explicit schema to reconstruct the source records for employees 117 and 118 from the reusable employee DataFrame. Keep both employee IDs and all source fields so the duplicated email remains visible.

**Expected output:** Two rows with employee IDs 117 and 118, identical email `FRAHMAN`, 11 columns, and the original decimal/date/integer types; no record is silently deduplicated.

## toDF()

### Question 4 ? Easy

**Business scenario:** A downstream workforce roster uses standard employee key and name labels.

**Problem statement:** Project employee ID, first name, and last name from `employees_df`, then use `toDF()` to rename them to `employee_key`, `given_name`, and `family_name`.

**Expected output:** A 20-row DataFrame with exactly those three columns in order; the original `employees_df` still has its original names.

### Question 5 ? Medium

**Business scenario:** The department dimension is being migrated to a warehouse naming convention.

**Problem statement:** Rename all columns of `departments_df` with `toDF()` to `department_key`, `department_label`, `manager_key`, and `location_key`. Validate the field count and row count.

**Expected output:** 18 rows and four renamed columns in the specified order; department 180 preserves NULL manager and location values.

### Question 6 ? Hard

**Business scenario:** An HR reporting extract joins employee and department data, where employee and department manager IDs can otherwise be ambiguous.

**Problem statement:** Left join `employees_df` to `departments_df` by `department_id`, then use `toDF()` to name all resulting columns: `employee_key`, `given_name`, `family_name`, `email`, `phone_number`, `hire_date`, `job_key`, `salary`, `commission_pct`, `employee_manager_key`, `department_key`, `department_name`, `department_manager_key`, `location_key`.

**Expected output:** A 20-row, 14-column result with the 14 unique names listed; employee 119 remains with NULL department fields.

## schema

### Question 7 ? Easy

**Business scenario:** The HR application team needs to confirm the employee master contract before loading it to a lakehouse.

**Problem statement:** Inspect `employees_df.schema` and identify the declared types and nullable flags for `employee_id`, `hire_date`, and `salary`.

**Expected output:** `employee_id` is non-nullable integer, `hire_date` is non-nullable date, and `salary` is non-nullable decimal(8,2).

### Question 8 ? Medium

**Business scenario:** A data-quality rule must confirm that optional employee relationships and commissions are represented as nullable fields.

**Problem statement:** Use `employees_df.schema` to validate the types and nullable flags for `commission_pct`, `manager_id`, and `department_id`.

**Expected output:** `commission_pct` is nullable decimal(4,2); `manager_id` and `department_id` are nullable integers.

### Question 9 ? Hard

**Business scenario:** The HR migration team needs a schema inventory for all seven Oracle HR entities before defining target contracts.

**Problem statement:** Build a schema inventory from the reusable DataFrames for EMPLOYEES, DEPARTMENTS, JOBS, JOB_HISTORY, LOCATIONS, COUNTRIES, and REGIONS. Include table name, field name, Spark type, and nullable flag.

**Expected output:** An inventory row for every field in all seven schemas; employee key is integer, hire/start/end dates are date, salaries are decimal, and country/job keys are strings.

## printSchema()

### Question 10 ? Easy

**Business scenario:** A department administrator wants a readable view of the location dimension structure.

**Problem statement:** Call `printSchema()` for `locations_df`.

**Expected output:** The schema tree prints six fields, including integer `location_id`, nullable address/postal/state fields, and string `country_id`.

### Question 11 ? Medium

**Business scenario:** An HR audit compares the employee contract with historical job assignment records.

**Problem statement:** Print the schemas of `employees_df` and `job_history_df`.

**Expected output:** Two readable schema trees are printed; employee salary and commission are decimals, employee hire date and job-history dates are dates, and employee/job/department keys use the declared integer/string types.

### Question 12 ? Hard

**Business scenario:** A migration engineer must inspect all Oracle HR source structures before generating a data dictionary.

**Problem statement:** Call `printSchema()` on each of the seven reusable HR DataFrames without changing any DataFrame.

**Expected output:** Seven schema trees are printed, one per HR entity, and all DataFrames remain available with their original columns and types.

## columns

### Question 13 ? Easy

**Business scenario:** A manager directory should contain only employee key, name, job, and department fields.

**Problem statement:** Use `employees_df.columns` to inspect available fields, then select employee ID, first name, last name, job ID, and department ID in that order.

**Expected output:** A 20-row projection with exactly `employee_id`, `first_name`, `last_name`, `job_id`, and `department_id`; employee 119 has a NULL department ID.

### Question 14 ? Medium

**Business scenario:** An employee-data contract requires a set of mandatory relationship and compensation fields.

**Problem statement:** Validate that `employee_id`, `manager_id`, `department_id`, `job_id`, and `commission_pct` are present in `employees_df.columns`; report any missing names.

**Expected output:** The missing-column list is empty; optional manager, department, and commission fields remain part of the schema despite containing NULL values.

### Question 15 ? Hard

**Business scenario:** A workforce report joins employee and department records and must produce an unambiguous output contract.

**Problem statement:** Left join `employees_df` and `departments_df` on `department_id`; use column-name metadata to project employee ID, employee first/last name, employee manager ID, department ID/name, department manager ID, and location ID in that order.

**Expected output:** The output has eight uniquely named columns, retains 20 employee rows, and preserves employee 119 with NULL department name and location.

## dtypes

### Question 16 ? Easy

**Business scenario:** Compensation reporting must verify salary precision and hire-date representation before export.

**Problem statement:** Inspect `employees_df.dtypes` and retrieve the runtime types of `salary` and `hire_date`.

**Expected output:** `salary` is `decimal(8,2)` and `hire_date` is `date`.

### Question 17 ? Medium

**Business scenario:** A regional HR hierarchy feed needs the type of each region and country key checked before a join.

**Problem statement:** Use `regions_df.dtypes` and `countries_df.dtypes` to validate `region_id`, `country_id`, and the country-to-region key.

**Expected output:** Both region ID fields are integer, `country_id` is string, and region/country name fields are strings.

### Question 18 ? Hard

**Business scenario:** The HR source-to-target mapping must detect type mismatches across primary and foreign keys in every entity.

**Problem statement:** Create a type-check report from the seven DataFrames' `dtypes` for employee/department/job/location/country/region keys and the salary/date fields. Flag any field that violates its Oracle-HR-compatible contract.

**Expected output:** A field-level report covers the requested keys and measures; employee/department/location/region IDs are integer, job/country IDs are string, salary fields are decimal, and hire/history dates are date with no mismatches.

## describe()

### Question 19 ? Easy

**Business scenario:** Compensation analysts need a quick profile of employee salaries before preparing a workforce report.

**Problem statement:** Run `describe()` on `employees_df.salary`.

**Expected output:** The result has `summary` and `salary` columns with count, mean, standard deviation, minimum, and maximum; count is 20, minimum is 2500.00, and maximum is 24000.00.

### Question 20 ? Medium

**Business scenario:** HR leadership wants a quick profile of salaries and sales commissions while checking how sparse commission data is.

**Problem statement:** Use `describe()` for employee salary and commission percentage, and inspect count, min, and max rows.

**Expected output:** The result includes both measures; salary count is 20, commission count is 3, and commission minimum/maximum are 0.10 and 0.15.

### Question 21 ? Hard

**Business scenario:** The IT workforce analyst wants to profile compensation for one department while retaining valid low/high salary bounds.

**Problem statement:** Filter `employees_df` to department 60 and use `describe()` on salary. Validate the number of employees represented and the salary endpoints.

**Expected output:** The profile represents five employees, with minimum salary 4200.00 and maximum salary 9000.00; NULL handling does not turn missing values into zero.

## summary()

### Question 22 ? Easy

**Business scenario:** A workforce migration report needs count and salary bounds using the configurable summary API.

**Problem statement:** Run `summary('count', 'min', 'max')` on `employees_df.salary`.

**Expected output:** Three summary rows are returned; salary count is 20, minimum is 2500.00, and maximum is 24000.00.

### Question 23 ? Medium

**Business scenario:** Compensation policy validation needs to review minimum and maximum pay ranges across job families.

**Problem statement:** Use `summary('count', 'min', 'max')` on `jobs_df.min_salary` and `jobs_df.max_salary`.

**Expected output:** Both job salary columns have count 18; the minimum of `min_salary` is 2000.00 and the maximum of `max_salary` is 40000.00.

### Question 24 ? Hard

**Business scenario:** HR migration controls must profile salaries only for employees assigned to a department while retaining duplicate email records.

**Problem statement:** Filter `employees_df` to non-NULL department IDs, then run `summary('count', 'min', 'max')` on salary. Separately validate the filtered row count and ensure the two FRAHMAN email records are not removed by the profile operation.

**Expected output:** The filtered set has 19 rows; salary summary count is 19, minimum is 2500.00, maximum is 24000.00, and the two distinct employee IDs with email FRAHMAN remain in the source.

## 7. Final practice completion checklist

- [ ] I completed all 24 question cells independently.
- [ ] I preserved HR primary and foreign key relationships while transforming data.
- [ ] I checked NULLs and duplicate business identifiers explicitly.
- [ ] I confirmed decimal compensation values and date/key data types.
- [ ] I retained the original reusable DataFrames for subsequent exercises.

After completing your work, stop the Spark session with `spark.stop()` in your own cell.